In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

In [14]:
# Define the data and indicators
class MomentumTrader:
    def __init__(self, df):
        self.df = df
        self.short_window = 10
        self.long_window = 30
        self.rsi_window = 14

    # Calculate MACD
    def calculate_macd(self):
        self.df['short_ma'] = self.df['Close'].rolling(window=self.short_window).mean()
        self.df['long_ma'] = self.df['Close'].rolling(window=self.long_window).mean()
        self.df['macd'] = self.df['short_ma'] - self.df['long_ma']
        return self.df

    # Calculate RSI
    def calculate_rsi(self):
        delta = self.df['Close'].diff().dropna()
        gains, losses = delta.copy(), delta.copy()
        gains[gains < 0] = 0
        losses[losses > 0] = 0
        avg_gain = gains.rolling(window=self.rsi_window).mean()
        avg_loss = abs(losses).rolling(window=self.rsi_window).mean()
        self.df['rsi'] = (avg_gain / avg_loss) * 100
        return self.df

    # Determine buy and sell signals based on MACD and RSI indicators
    def determine_signals(self):
        self.df['signal'] = np.where((self.df['macd'] > 0) & (self.df['rsi'] < 30), 'Buy', 
                                      np.where((self.df['macd'] < 0) & (self.df['rsi'] > 70), 'Sell', 'No Signal'))
        return self.df

    # Generate trading signals for the backtesting period
    def generate_signals(self, start_date, end_date):
        trading_df = self.df[(self.df['Date'] >= start_date) & (self.df['Date'] <= end_date)]
        trading_df = trading_df.sort_values(by='Date')
        return trading_df

In [35]:
# Load the data
df = pd.DataFrame({
    'Close': [100, 110, 120, 130, 140, 150, 160, 170, 180, 190],
    'Open': [90, 100, 110, 120, 130, 140, 150, 160, 170, 180],
    'High': [105, 115, 125, 135, 145, 155, 165, 175, 185, 195],
    'Low': [95, 105, 115, 125, 135, 145, 155, 165, 175, 185]
})
df

,Close,Open,High,Low
0,100,90,105,95
1,110,100,115,105
2,120,110,125,115
3,130,120,135,125
4,140,130,145,135
5,150,140,155,145
6,160,150,165,155
7,170,160,175,165
8,180,170,185,175
9,190,180,195,185


In [53]:
df = pd.read_csv('./datas/samsung.csv')
df = df[['Close', 'Open', 'High', 'Low', 'Date']]

In [54]:
# Initialize the Momentum Trader
mt = MomentumTrader(df)

In [55]:
# Calculate MACD and RSI indicators
df = mt.calculate_macd()
df = mt.calculate_rsi()

In [56]:
# Determine trading signals based on MACD and RSI indicators
df = mt.determine_signals()

In [57]:
# Define the backtesting period
start_date = '2023-01-02'
end_date = '2023-12-28'

In [58]:
# Generate trading signals for the backtesting period
backtesting_df = mt.generate_signals(start_date, end_date)

In [59]:
backtesting_df

,Close,Open,High,Low,Date,short_ma,long_ma,macd,rsi,signal
0,55500,55500,56100,55200,2023-01-02,NaN,NaN,NaN,NaN,No Signal
1,55400,55400,56000,54500,2023-01-03,NaN,NaN,NaN,NaN,No Signal
2,57800,55700,58000,55600,2023-01-04,NaN,NaN,NaN,NaN,No Signal
3,58200,58200,58800,57600,2023-01-05,NaN,NaN,NaN,NaN,No Signal
4,59000,58300,59400,57900,2023-01-06,NaN,NaN,NaN,NaN,No Signal
...,...,...,...,...,...,...,...,...,...,...
240,75000,74600,75000,74300,2023-12-21,73440.0,72483.333333,956.666667,211.111111,No Signal
241,75900,75800,76300,75400,2023-12-22,73770.0,72663.333333,1106.666667,222.222222,No Signal
242,76600,76100,76700,75700,2023-12-26,74130.0,72870.000000,1260.000000,515.384615,No Signal
243,78000,76700,78000,76500,2023-12-27,74580.0,73110.000000,1470.000000,584.615385,No Signal
